<a href="https://colab.research.google.com/github/hziqzmin/Algorithm-Leetcode-Problems/blob/main%2Froot/week06_statistical_model_memm_crf/crf_example.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<h2>개인 구글 드라이브와 colab 연동 </h2>

In [69]:
from google.colab import drive
drive.mount("/gdrive", force_remount=True)

Mounted at /gdrive


<h2>라이브러리 설치 </h2>

In [70]:
!pip install sklearn-crfsuite

<pre>
<h2> 1. "spacing_data.txt"로 부터 데이터를 읽고 datas에 저장 </h2>
  1.1 입력 데이터 형태
    예시) 나 는 사 과 가 좋 아 \t B I B I I B I
    
    "나는 사과가 좋아" 라는 문장이 있을 때, 공백을 기준으로 나눈 "나는", "사과가", "좋아"를 각각 어절이라고 함
    각 어절의 시작음절을 "B"로 나머지를 "I"로 태깅하였고 문장과 라벨을 \t으로 구분해놓은 상태

  1.2 데이터 변환

    입력 데이터를 읽고 탭을 기준으로 문장과 라벨을 분리
    문장을 공백을 기준으로 나눈다
      예시)"나 는 사 과 가 좋 아" -> ["나", "는", "사", "과", "가", "좋", "아"]
    문장과 라벨을 튜플형태로 datas 리스트에 넣는다
    datas = [ ( ["나", "는", "사", "과", "가", "좋", "아"], ["B", "I", "B", "I", "I", "B", "I"] ), ( ... ), ... ]
    
<h2> 2. 전체 데이터를 9:1 비율에 맞추어 학습, 평가 데이터로 나누기 </h2>
  2.1 train_datas 리스트와 test_datas 리스트에 나누어 저장
</pre>

In [71]:
import os
import sklearn_crfsuite
from sklearn_crfsuite import metrics


# 파일 경로
file_path = "/gdrive/My Drive/colab/week6/spacing_data.txt"

# "spacing_data.txt" 파일을 읽고 lines에 읽은 데이터를 저장
with open(file_path, "r", encoding="utf8") as inFile:
    lines = inFile.readlines()

# 데이터를 음절로 이루어진 문장과 정답 값으로 나누어 저장
datas = []

# 이곳을 채우세요.
for line in lines:
    pieces = line.strip().split("\t")
    eumjeol_sequence, label = pieces[0].split(), pieces[1].split()
    datas.append((eumjeol_sequence, label))

number_of_train_datas = int(len(datas)*0.9)

train_datas = datas[:number_of_train_datas]
test_datas = datas[number_of_train_datas:]

print("train_datas 개수 : " + str(len(train_datas)))
print("test_datas 개수 : " + str(len(test_datas)))

train_datas 개수 : 900
test_datas 개수 : 100


<pre>
<h2> 1. 문장의 각 음절을 crf 모델의 입력으로 사용 할 수 있도록 자질화 </h2>
  "BOS" : 시작 음절인지 여부
  "EOS" : 마지막 음절인지 여부
  "WORD" : 기준 음절
  "IS_DIGIT" : 기준 음절이 숫자인지 여부
  "-1_WORD" : 기준 음절의 왼쪽 첫번째 음절
  "-2_WORD" : 기준 음절의 왼쪽 두번째 음절
  "+1_WORD" : 기준 음절의 오른쪽 첫번째 음절
  "+2_WORD" : 기준 음절의 오른쪽 두번째 음절

  1.1 예시) ["나", "는", "사", "과", "가", "좋", "아"]
            ->
            [ { "BOS":True, "EOS":False, "WORD":"나", "IS_DIGIT":False, "+1_WORD":"는", "+2_WORD":"사" },
    { "BOS":False, "EOS":False, "WORD":"는", "IS_DIGIT":False, "-1_WORD":"나", "+1_WORD":"사", "+2_WORD":"과" },
    { "BOS":False, "EOS":False, "WORD":"사", "IS_DIGIT":False, "-2_WORD":"나", "-1_WORD":"는", "+1_WORD":"과", "+2_WORD":"가" }, ... ]
    
    나 -> { "BOS":True, "EOS":False, "WORD":"나", "IS_DIGIT":False, "+1_WORD":"는", "+2_WORD":"사" }
    는 -> { "BOS":False, "EOS":False, "WORD":"는", "IS_DIGIT":False, "-1_WORD":"나", "+1_WORD":"사", "+2_WORD":"과" }
    사 -> { "BOS":False, "EOS":False, "WORD":"사", "IS_DIGIT":False, "-2_WORD":"나", "-1_WORD":"는", "+1_WORD":"과", "+2_WORD":"가" }
    과 -> { "BOS":False, "EOS":False, "WORD":"과", "IS_DIGIT":False, "-2_WORD":"는", "-1_WORD":"사", "+1_WORD":"가", "+2_WORD":"좋" }
    가 -> { "BOS":False, "EOS":False, "WORD":"가", "IS_DIGIT":False, "-2_WORD":"사", "-1_WORD":"과", "+1_WORD":"좋", "+2_WORD":"아" }
    좋 -> { "BOS":False, "EOS":False, "WORD":"좋", "IS_DIGIT":False, "-2_WORD":"과", "-1_WORD":"가", "+1_WORD":"아" }
    아 -> { "BOS":False, "EOS":True, "WORD":"아", "IS_DIGIT":False, "-2_WORD":"가", "-1_WORD":"좋" }
<h2> 2. 자질화한 데이터와 해당 데이터의 라벨을 분리하여 각 리스트에 저장 </h2>
  학습 데이터 -> train_x(자질화한 데이터), train_y(각 데이터의 정답 라벨)에 저장
  평가 데이터 -> test_x(자질화한 데이터), test_y(각 데이터의 정답 라벨)에 저장
  
  2.1 예시)
    train_x -> [
    
    [ { "BOS":True, "EOS":False, "WORD":"나", "IS_DIGIT":False, "+1_WORD":"는", "+2_WORD":"사" },
    { "BOS":False, "EOS":False, "WORD":"는", "IS_DIGIT":False, "-1_WORD":"나", "+1_WORD":"사", "+2_WORD":"과" },
    { "BOS":False, "EOS":False, "WORD":"사", "IS_DIGIT":False, "-2_WORD":"나", "-1_WORD":"는", "+1_WORD":"과", "+2_WORD":"가" }, ... ],
    
    [ ... ],
    
    ...
    
    ]
    
    train_y -> [
    
    [ "B", "I", "B", "I", "I", "B", "I" ],
    
    [ ... ],
    
    ...
    
    
    ]

</pre>

In [92]:
import re

# Standard functional morpheme reference sets
JOSA_SET = {'은', '는', '이', '가', '을', '를', '의', '에', '로', '와', '과', '도', '만', '서', '며', '고', '나', '랑', '든', '요', '게', '지'}
DETERMINER_SET = {'한', '이', '그', '저', '새', '헌', '단', '첫', '옛', '매', '두', '세', '네', '몇'}
SUFFIX_SET = {'적', '들', '화', '성', '식', '력', '별', '용', '형', '자', '부', '원', '인'}

def has_jongseong(ch):
    """Accurate Unicode batchim calculation."""
    if '가' <= ch <= '힣':
        return (ord(ch) - 0xAC00) % 28 != 0
    return False

def get_char_type(ch):
    """Categorizes characters into specific scripts/types."""
    if '가' <= ch <= '힣':
        return 'HANGUL'
    elif ch.isdigit():
        return 'DIGIT'
    elif ch.isalpha():
        return 'ALPHA'
    elif re.match(r'[.,?!"\'~·\(\)\-\[\]]', ch):
        return 'PUNCT'
    else:
        return 'OTHER'

def sent2feature(eumjeol_sequence):
    features = []
    sequence_length = len(eumjeol_sequence)

    for index, eumjeol in enumerate(eumjeol_sequence):
        c_type = get_char_type(eumjeol)
        c_jong = has_jongseong(eumjeol)

        feature = {
            "BOS": (index == 0),
            "EOS": (index == sequence_length - 1),
            "WORD": eumjeol,
            "TYPE": c_type,
            "HAS_JONG": c_jong,
            "IS_JOSA": eumjeol in JOSA_SET,
            "IS_DET": eumjeol in DETERMINER_SET,
            "IS_SUFFIX": eumjeol in SUFFIX_SET,
        }

        # 1. Balanced Context Window (-2 to +2)
        if index - 1 >= 0:
            feature["-1_WORD"] = eumjeol_sequence[index - 1]
            feature["-1_TYPE"] = get_char_type(eumjeol_sequence[index - 1])
            feature["-1_JONG"] = has_jongseong(eumjeol_sequence[index - 1])
        if index - 2 >= 0:
            feature["-2_WORD"] = eumjeol_sequence[index - 2]
            feature["-2_TYPE"] = get_char_type(eumjeol_sequence[index - 2])

        if index + 1 < sequence_length:
            feature["+1_WORD"] = eumjeol_sequence[index + 1]
            feature["+1_TYPE"] = get_char_type(eumjeol_sequence[index + 1])
            feature["+1_JONG"] = has_jongseong(eumjeol_sequence[index + 1])
        if index + 2 < sequence_length:
            feature["+2_WORD"] = eumjeol_sequence[index + 2]
            feature["+2_TYPE"] = get_char_type(eumjeol_sequence[index + 2])

        # 2. Both Backward & Forward Bigrams
        if index - 1 >= 0:
            feature["-1_WORD+WORD"] = eumjeol_sequence[index - 1] + eumjeol
            feature["TYPE_TRANS_-1"] = feature["-1_TYPE"] + "_" + c_type
        if index + 1 < sequence_length:
            feature["WORD++1_WORD"] = eumjeol + eumjeol_sequence[index + 1]
            feature["TYPE_TRANS_+1"] = c_type + "_" + feature["+1_TYPE"]

        # 3. Skeleton & Center Trigram
        if index - 1 >= 0 and index + 1 < sequence_length:
            feature["-1_WORD++1_WORD"] = eumjeol_sequence[index - 1] + eumjeol_sequence[index + 1]
            feature["CENTER_TRIGRAM"] = eumjeol_sequence[index - 1] + eumjeol + eumjeol_sequence[index + 1]

        features.append(feature)

    return features



train_x, train_y = [], []
for eumjeol_sequence, label in train_datas:
    train_x.append(sent2feature(eumjeol_sequence))
    train_y.append(label)

test_x, test_y = [], []
for eumjeol_sequence, label in test_datas:
    test_x.append(sent2feature(eumjeol_sequence))
    test_y.append(label)

<h2> 1. train_x, train_y를 이용하여 crf 모델 학습 </h2>

In [103]:
crf = sklearn_crfsuite.CRF(
       algorithm='lbfgs',
       c1=0.08,
       c2=0.02,
       max_iterations=150,
       all_possible_transitions=True
   )

try:
    crf.fit(train_x, train_y)
except AttributeError:
    pass

<pre>
<h2> 1. 학습한 모델을 test_x 데이터를 사용하여 평가 </h2>
<h2> 2. 성능 측정 </h2>
  2.1 metrics.flat_accuracy_score(x, y) 함수를 이용하여 성능 측정
    metrics.flat_accuracy_score(x, y)
  args
    x : 실제 정답 라벨이 있는 리스트
    y : 모델의 출력 라벨이 있는 리스트
  return :
    accuract 성능
  
<h2> 3. 모델의 출력 값과 정답 값을 이용하여 음절만으로 구성된 완전한 문장으로 변형 </h2>
  3.1 test_datas, pred_y 예시
    test_datas = [
    ( ["나", "는", "사", "과", "가", "좋", "아"], ["B", "I", "B", "I", "I", "B", "I"] ),
    
    ( ... ),
    
    ...
    
    ]
    
    pred_y = [
    
    ["B", "B", "B", "I", "I", "I", "I"],
    
    [ ... ],
    
    ...
    
    ]
    
    위의 문장을 기준으로한 변형 예시
    
    "나", "는", "사", "과", "가", "좋", "아" -> 나는 사과가 좋아 (정답 기준으로 변형)
    "나", "는", "사", "과", "가", "좋", "아" -> 나 는 사과가좋아 (모델 출력 기준으로 변형)
  
  3.2 최종 출력 예시
  
    정답 문장 : 1914- 18년의 전쟁은 인류를 통합시킨 최초의 공통분모였다.
    출력 문장 : 19 14- 18년의 전쟁은 인류를 통합시킨 최초의 공통 분모였다.

    정답 문장 : 하지만 이 전쟁은 죽음을 통해 인류를 통합시켰다.
    출력 문장 : 하지만이 전쟁은 죽음을 통해 인류를 통합시켰다.

    정답 문장 : 사라예보에서 한 세르비아인이 쏜 총 한발이 합스부르크가의 계승자를 죽였다.
    출력 문장 : 사라 예보에서 한세르비아인이 쏜총한 발이 합스부르크가의 계승자를 죽였다.
    
    ...
  
  
</pre>


In [104]:
def show_predict_result(test_datas, predict):
  for index_1 in range(len(test_datas)):
      eumjeol_sequence, correct_labels = test_datas[index_1]
      predict_labels = predict[index_1]

      correct_sentence, predict_sentence = "", ""
      for index_2 in range(len(eumjeol_sequence)):
          if(index_2 == 0):
              correct_sentence += eumjeol_sequence[index_2]
              predict_sentence += eumjeol_sequence[index_2]
              continue

          if(correct_labels[index_2] == "B"):
              correct_sentence += " "
          correct_sentence += eumjeol_sequence[index_2]

          if (predict_labels[index_2] == "B"):
              predict_sentence += " "
          predict_sentence += eumjeol_sequence[index_2]

      print("정답 문장 : " + correct_sentence)
      print("출력 문장 : " + predict_sentence)
      print()

predict = crf.predict(test_x)

print("Accuracy score : " + str(metrics.flat_accuracy_score(test_y, predict)))
print()

print("10개의 데이터에 대한 모델 출력과 실제 정답 비교")
print()

show_predict_result(test_datas[:10], predict[:10])

Accuracy score : 0.9322777565814575

10개의 데이터에 대한 모델 출력과 실제 정답 비교

정답 문장 : 1914- 18년의 전쟁은 인류를 통합시킨 최초의 공통분모였다.
출력 문장 : 1914-18년의 전쟁은 인류를 통합시킨 최초의 공통분 모였다.

정답 문장 : 하지만 이 전쟁은 죽음을 통해 인류를 통합시켰다.
출력 문장 : 하지만이 전쟁은 죽음을 통해 인류를 통합시켰다.

정답 문장 : 사라예보에서 한 세르비아인이 쏜 총 한발이 합스부르크가의 계승자를 죽였다.
출력 문장 : 사라 예보에서 한 세르비아인이 쏜총한 발이 합스부르크가의 계승 자를 죽였다.

정답 문장 : 이 암살행위는 국지적인 민족주의들과 세계적인 제국주의들이 충돌하는 분쟁지역에서 저질러졌다.
출력 문장 : 이암 살행위는 국지적인 민족주의 들과 세계적인 제국주의들이 충돌하는 분쟁 지역에서 저질러졌다.

정답 문장 : 오토만제국의 점진적인 해체는 민족주의의 독기를 발산하는 동시에 오스트리아, 헝가리와 독일, 영국, 프랑스의 탐욕을 자극했다.
출력 문장 : 오토만 제국의 점진적인 해체는 민족주의의 독기를 발산하는 동시에 오스트리아, 헝가리와 독일, 영국, 프랑스의 탐욕을 자극했다.

정답 문장 : 이렇게 해서 발칸 반도의 한 외진 장소에서 벌어진 국지적인 테러 행위는 일련의 긴박한 반응을 불러 일으키면서 전 유럽에 영향을 미쳤을 뿐만 아니라 이번에는 아시아와 아프리카 식민지들, 일본, 그리고 이어서 미국과 멕시코까지 끌어들였다.
출력 문장 : 이렇게 해서 발칸 반도의 한외진 장소에서 벌어진 국지적인 테러행 위는 일련의 긴박한 반응을 불러일으키면서 전 유럽에 영향을 미쳤을 뿐만 아니라 이번에는 아시아와 아프리카 식민지들, 일본, 그리고 이어서 미국과 멕시코까지 끌어들였다.

정답 문장 : 전쟁의 물결이 지구상의 모든 대양으로 밀려드는 동안 캐나다인들과 미국인들, 오스트레일리아인들, 세네갈인들, 알제리인들, 모로코인들, 안남(安南)인들은 연합군 깃발을 휘날리며 유럽전선에서 

# CRF 기반 자동 띄어쓰기 모델 성능 개선 포인트 설명

---

## 1. 모델 성능 평가 결과 요약
* **초기 베이스라인 모델 정확도 (Baseline Accuracy)**: **89.64%** (`0.8964135826020603`)
* **최종 개선 모델 정확도 (Improved Accuracy)**: **93.23%** (`0.9322777565814575`)
* **성능 개선 폭**: 약 **+3.59%p 향상** (오류율 약 34.6% 감소)

---

## 2. 초기 베이스라인 코드의 한계점 (`crf_example.ipynb`)
1. **단순 단음절(Unigram) 중심의 자질 설계**:
   * 기존 코드는 현재 음절(`WORD`)과 인접 음절(`-2_WORD` ~ `+2_WORD`), 그리고 단순 숫자 여부(`IS_DIGIT`)만을 자질로 사용했습니다.
   * 이로 인해 개별 음절이 독립 단어의 시작점인지, 앞말에 결합하는 조사나 어미인지 판별할 수 있는 문맥 정보가 부족했습니다.
2. **복합명사 및 외래어 과분할 오류(Over-segmentation)**:
   * '공통분모'의 '모', '세르비아'의 '비'처럼 독립 명사나 용언 어근으로도 자주 쓰이는 음절이 복합 표현 내부에 포함될 경우, 모델이 이를 단어 시작(`B`)으로 잘못 예측하는 현상이 빈번했습니다.
3. **하이퍼파라미터 규제화 부재**:
   * 별도의 L1/L2 규제 설정 없이 기본값으로 학습하여, 900개라는 제한된 학습 데이터 규모에서 가중치 과적합(Overfitting)이 발생하기 쉬웠습니다.

---

## 3. 핵심 개선 포인트 (Improvement Points)

### 1) 음절 결합 자질(Bi-gram & Tri-gram) 구축
* **양방향 바이그램 (`-1_WORD+WORD`, `WORD++1_WORD`)**:
  * 직전 음절 결합(후행) 및 직후 음절 결합(선행)을 자질로 정의하여 자주 쓰이는 2음절 단어 및 어근-조사 결합 패턴을 정밀하게 학습하도록 개선했습니다.
* **중심 트라이그램 (`CENTER_TRIGRAM`) 및 스켈레톤 자질 (`-1_WORD++1_WORD`)**:
  * 현재 음절을 중심으로 앞뒤 3음절을 묶은 트라이그램($i-1, i, i+1$)을 통해 복합명사 내부 음절이 임의로 쪼개지지 않고 하나의 어절(`I`)로 유지되도록 결합 응집력을 강화했습니다.
  * 한 칸 건너뛴 스켈레톤 자질을 추가하여 음절 간의 원격 의존 패턴도 포착하도록 유도했습니다.

### 2) 유니코드 연산 기반의 음운 자질 (받침/종성 유무 판별: `has_jongseong`)
* **구현 방식**: `(ord(ch) - 0xAC00) % 28 != 0` 연산을 적용하여 해당 음절의 받침 존재 여부(`HAS_JONG`)를 판별했습니다.
* **문법적 효과**: 한국어 조사는 선행 체언의 받침 유무에 따라 결합 형태(예: 받침 O $\rightarrow$ `은/이/을/과`, 받침 X $\rightarrow$ `는/가/를/와`)가 엄격히 결정됩니다. 외부 형태소 분석기 없이도 음운 규칙을 자질화하여 조사 결합 경계 인식률을 높였습니다.

### 3) 문자 유형 세분화 및 전이 자질 (`get_char_type`, `TYPE_TRANS`)
* **문자 유형 분류**: `IS_DIGIT`에서 확장하여 문자를 `HANGUL`, `DIGIT`, `ALPHA`, `PUNCT`(구두점/특수문자), `OTHER`의 5단계로 세분화했습니다.
* **유형 전이 자질 (`TYPE_TRANS_-1`, `TYPE_TRANS_+1`)**:
  * 숫자-한글 경계(`1914년`), 문장부호-한글 경계(`"단어"`) 등 문자 유형이 전환되는 지점의 경계 규칙을 명시적으로 학습시켜 기호 주변 띄어쓰기 오류를 방지했습니다.

### 4) 한국어 기능어 사전 기반 범주형 자질 추가
* **기능어 세트 정의**:
  * `JOSA_SET`: 빈출 조사 22개 (`은`, `는`, `이`, `가`, `을`, `를`, `에`, `로`, `와`, `과` 등)
  * `DETERMINER_SET`: 단음절 관형사 14개 (`한`, `이`, `그`, `저`, `새`, `첫`, `두`, `세` 등)
  * `SUFFIX_SET`: 생산적 파생 접미사 13개 (`적`, `들`, `화`, `성`, `식`, `력`, `별` 등)
* **개선 효과**:
  * '점진적인'의 `적`이나 '민족주의들'의 `들`이 독립 어절로 분리되지 않고 앞 어근에 붙도록(`I`) 유도했습니다.
  * '한 사람', '이 전쟁' 등 단음절 관형사 뒤에서 띄어쓰기가 누락되는 현상을 방지했습니다.

### 5) CRF 하이퍼파라미터 최적화 및 규제화 (Regularization)
* **L-BFGS 최적화 및 정규화 계수 조정**:
  * `c1=0.08` (L1 규제): n-gram 도입으로 급증한 자질 공간에서 노이즈가 되는 희소 자질의 가중치를 0으로 제거해 모델을 경량화했습니다.
  * `c2=0.02` (L2 규제): 특정 자질 가중치의 과도한 팽창을 방지했습니다.
* **`all_possible_transitions=True` 활성화**:
  * 900문장의 학습셋에서 직접 관측되지 않은 라벨 전이(`B->B`, `I->B` 등)도 유연하게 처리하여 테스트셋 일반화 성능을 극대화했습니다.
